# 28 — wall-resolution test at $\bar\lambda=0.069$, $\gamma_{ij}=4$

The $N_b=2$ lattice/reduced $I_\mathcal{K}$ ratio for $\bar\lambda=0.069$ drifts down at late times, while $\bar\lambda=0.84$ converges to 1. Low-$\gamma_{ij}$ pairs are evaluated at very late times when the surrogate is used at large $\gamma_*$, so we check whether the pair library is resolved there.

One $\gamma_{ij}=4$ row, set up exactly like a row of the production scan `runtime_scan_lb0.069_gs1-20_kR1-100_nw32` (same $T_{\max}$, time grid, $\omega$ grid, $N_{\min}$, panels per oscillation, `how_often_ds`), run at `wall_points` = 10, 20 (production), 30, 40. `wall_points` sets $dz$ = (wall width at collision)/`wall_points`. One Perlmutter GPU node, one variant per GPU.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.069
GAMMA_IJ   = 4.0
WALL_POINTS = [10, 20, 30, 40]

# Production-scan definition (generate_surrogate defaults, as used for runtime_scan_lb0.069_gs1-20_kR1-100_nw32)
GAMMA_STAR_MIN, GAMMA_STAR_MAX = 1.0, 20.0
KR_MIN, KR_MAX = 1.0, 100.0
N_OMEGA, N_K = 32, 51
PAIR_GAMMA_FACTOR, TIME_FACTOR, OMEGA_MIN_GAMMA_FACTOR = 2.5, 1.5, 1.5
N_MIN, PPW, HOW_OFTEN_DS, S_BATCH_SIZE = 128, 40, 5, 128

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))
kin = model.kinematics

def time_at_gamma(gamma):
    return 0.0 if gamma <= 1.0 else brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)

def rstar_at_gamma(gamma):
    return 2.0 * float(kin.R(time_at_gamma(gamma), r='mid'))

dt        = 2.0 * model.instanton.rmid_0 / 6.9364
t_max     = TIME_FACTOR * rstar_at_gamma(GAMMA_STAR_MAX)
t_first   = time_at_gamma(GAMMA_IJ)
n_t       = int(np.ceil((t_max - t_first) / dt)) + 1
times     = np.linspace(t_first, t_max, n_t)
omega_min = KR_MIN / rstar_at_gamma(OMEGA_MIN_GAMMA_FACTOR * GAMMA_STAR_MAX)
omega_max = KR_MAX / rstar_at_gamma(max(GAMMA_STAR_MIN, GAMMA_IJ / PAIR_GAMMA_FACTOR))
omega     = np.geomspace(omega_min, omega_max, N_OMEGA)

CHECK_ROOT  = ROOT / f'data/wall_points_check_lb{LAMBDA_BAR:g}_g{GAMMA_IJ:g}'
setup_dir   = CHECK_ROOT / 'setups'
output_root = CHECK_ROOT / 'gpu'
setup_dir.mkdir(parents=True, exist_ok=True); output_root.mkdir(parents=True, exist_ok=True)
print(f't_first={t_first:.3f}  T_MAX={t_max:.3f}  n_t={n_t}  omega={omega_min:.4g}..{omega_max:.4g}')

t_first=4.041  T_MAX=62.875  n_t=206  omega=0.0159..30.97


In [2]:
rows = []
for wp in WALL_POINTS:
    params = BubbleMasterParams(dz=None, wall_points=wp, n_w=N_OMEGA, n_k=N_K, how_often_ds=HOW_OFTEN_DS,
                                baby_steps=100, cutoff_type=0, t_0_scal=1.0, collision_radius='mid')
    setup_path = setup_dir / f'wp{wp}.h5'
    write_2d_setup(model, GAMMA_IJ, times, setup_path, params, wlist=omega)
    with h5py.File(setup_path) as f:
        print(f'wp={wp:2d}: dz={f.attrs["dz_target"]:.5f}  ds={f.attrs["ds"]:.5f}  n_z={f.attrs["n_z"]}  d={f.attrs["d"]:.3f}  smax={f.attrs["smax"]:.2f}')
    rows.append((wp, setup_path.relative_to(ROOT), f'wp{wp}'))

manifest = CHECK_ROOT / 'manifest.tsv'
manifest.write_text('wall_points\tsetup\tname\n' + ''.join(f'{wp}\t{s}\t{n}\n' for wp, s, n in rows))

wp=10: dz=0.02004  ds=0.00401  n_z=3760  d=8.324  smax=62.88
wp=20: dz=0.01002  ds=0.00200  n_z=7521  d=8.324  smax=62.88
wp=30: dz=0.00668  ds=0.00134  n_z=11282  d=8.324  smax=62.88
wp=40: dz=0.00501  ds=0.00100  n_z=15042  d=8.324  smax=62.88


251

In [3]:
GPU_ACCOUNT  = 'm3166_g'
GPU_WALLTIME = '04:00:00'   # production wp=20 row took ~20 min; cost grows ~wp^2 (dz and ds both shrink), so wp=40 ~1.5 h

script_path = ROOT / 'scripts' / f'wall_points_check_lb{LAMBDA_BAR:g}_g{GAMMA_IJ:g}_gpu.sh'
(ROOT / 'logs').mkdir(exist_ok=True)
script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_wp_lb{LAMBDA_BAR:g}_g{GAMMA_IJ:g}
#SBATCH --output={ROOT}/logs/bm_wp_lb{LAMBDA_BAR:g}_g{GAMMA_IJ:g}_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={GPU_WALLTIME}
set -euo pipefail
REPO_ROOT="{ROOT}"
cd "$REPO_ROOT"
pids=()
while IFS=$'\\t' read -r wp setup name; do
    [[ "$wp" == "wall_points" ]] && continue
    output="{output_root.relative_to(ROOT)}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {N_MIN} --filon-panels-per-osc {PPW} --s-batch-size {S_BATCH_SIZE} &
    pids+=("$!")
done < {manifest.relative_to(ROOT)}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
''')
script_path.chmod(0o755)
print(f'Written: {script_path.relative_to(ROOT)}\nSubmit: sbatch {script_path.relative_to(ROOT)}')

Written: scripts/wall_points_check_lb0.069_g4_gpu.sh
Submit: sbatch scripts/wall_points_check_lb0.069_g4_gpu.sh


## Results

Spectra at every requested time, each variant relative to `wall_points=40`.

In [4]:
def load_row(out_dir):
    files = sorted(Path(out_dir).glob('result_*.h5'), key=lambda p: int(p.stem.split('_')[1]))
    if not files:
        raise FileNotFoundError(f'no result_*.h5 in {out_dir}')
    t, S = [], []
    for p in files:
        with h5py.File(p) as f:
            t.append(float(f.attrs['t'])); S.append(f['spectrum'][:]); w = f['w'][:]
    return np.array(t), w, np.array(S)

res = {}
for wp, _, name in rows:
    try:
        res[wp] = load_row(output_root / name)
    except Exception as e:
        print(f'wp={wp}: not ready')

if 40 in res:
    t_ref, w, S_ref = res[40]; d = 2 * float(kin.R(time_at_gamma(GAMMA_IJ), r='mid'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for wp, (t, _, S) in res.items():
        if wp == 40: continue
        n = min(len(t), len(t_ref))
        E, E_ref = np.trapz(S[:n], np.log(w), axis=1), np.trapz(S_ref[:n], np.log(w), axis=1)
        axes[0].plot(t[:n] / d, E / E_ref, label=f'wall_points={wp}')
        axes[1].semilogx(w * d, S[n - 1] / S_ref[n - 1], label=f'wall_points={wp}')
    axes[0].set_xlabel('$t/d$'); axes[0].set_ylabel('band-integrated / (wall_points=40)')
    axes[1].set_xlabel(r'$\omega d$'); axes[1].set_ylabel('spectrum / (wall_points=40), last time')
    for a in axes: a.axhline(1, color='0.6', ls=':'); a.legend(frameon=False)
    plt.tight_layout(); plt.show()

wp=10: not ready
wp=20: not ready
wp=30: not ready
wp=40: not ready
